## 上下文 -

### docs

### styling

### directory

In [ ]:
#| hide
!tree ..

..
├── LICENSE
├── MANIFEST.in
├── README.md
├── _proc
│   ├── 00_core.ipynb
│   ├── _docs
│   │   ├── index.html
│   │   ├── robots.txt
│   │   └── sitemap.xml
│   ├── _quarto.yml
│   ├── index.ipynb
│   ├── nbdev.yml
│   └── styles.css
├── nbs
│   ├── 00_core.ipynb
│   ├── _quarto.yml
│   ├── db.db
│   ├── db.db-shm
│   ├── db.db-wal
│   ├── index.ipynb
│   ├── nbdev.yml
│   └── styles.css
├── pyproject.toml
├── vlm_monitor
│   ├── __init__.py
│   ├── __pycache__
│   │   ├── __init__.cpython-312.pyc
│   │   └── core.cpython-312.pyc
│   ├── _modidx.py
│   └── core.py
└── vlm_monitor.egg-info
    ├── PKG-INFO
    ├── SOURCES.txt
    ├── dependency_links.txt
    ├── entry_points.txt
    ├── requires.txt
    └── top_level.txt

7 directories, 31 files


# core

> Fill in a module description here

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp core

## Database

In [ ]:
#| export
from fastcore.all import *

In [ ]:
#| export
class Video: id:int; title:str=''; overview:str=''; transcript:str=''; length:int=0; sample_rate:int=0
class Frame: id:int; video_id:int; frame_number:int; subtitle:str=''
class Run: id:int; deploy_time:str; finish_time:str; request_timer:str; video_id:int; model:str; usage:str; num_frames:int; description:str
class RunFrame: run_id:int; frame_id:int; type:str; system_prompt:str; prompt:str; description:str; usage:str

In [ ]:
#| export
from fastlite import *

In [ ]:
!rm db.db
db = database('db.db'); db

<Database <apsw.Connection "/app/data/vlm-monitor/nbs/db.db">>

In [ ]:
??Queryable.schema

Type:        property
String form: <property object>
Source:     
# Queryable.schema.fget
@property
def schema(self) -> str:
    "SQL schema for this table or view."
    return self.db.execute(
        "select sql from sqlite_master where name = ?", (self.name,)
    ).fetchone()[0]

In [ ]:
@patch(as_prop=True)
def schema(self:Queryable) -> str:
    "SQL schema for this table or view."
    return hl_md(self.db.execute(
        "select sql from sqlite_master where name = ?", (self.name,)
    ).fetchone()[0], lang='sql')

In [ ]:
videos = db.create(Video, transform=True); videos.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [video] (
   [id] INTEGER PRIMARY KEY,
   [title] TEXT,
   [overview] TEXT,
   [transcript] TEXT,
   [length] INTEGER,
   [sample_rate] INTEGER
)
```

</div>

In [ ]:
frames = db.create(Frame, transform=True, foreign_keys=[('video_id', 'video', 'id')]); frames.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [frame] (
   [id] INTEGER PRIMARY KEY,
   [video_id] INTEGER REFERENCES [video]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [frame_number] INTEGER,
   [subtitle] TEXT
)
```

</div>

In [ ]:
runs = db.create(Run, transform=True); runs.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [run] (
   [id] INTEGER PRIMARY KEY,
   [deploy_time] TEXT,
   [finish_time] TEXT,
   [request_timer] TEXT,
   [video_id] INTEGER,
   [model] TEXT,
   [usage] TEXT,
   [num_frames] INTEGER,
   [description] TEXT
)
```

</div>

In [ ]:
runframes = db.create(RunFrame, pk=['run_id', 'frame_id', 'type'], foreign_keys=[('run_id', 'run', 'id'), ('frame_id', 'frame', 'id')], transform=True); runframes.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [run_frame] (
   [run_id] INTEGER REFERENCES [run]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [frame_id] INTEGER REFERENCES [frame]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [type] TEXT,
   [system_prompt] TEXT,
   [prompt] TEXT,
   [description] TEXT,
   [usage] TEXT,
   PRIMARY KEY ([run_id], [frame_id], [type])
)
```

</div>

In [ ]:
#| export
from typing import NamedTuple
from apswutils.db import Database, Table
class DBResources(NamedTuple): db:Database; videos:Table; frames:Table; runs:Table; runframes:Table

In [ ]:
#| export

def init_db(
    path:str|Path='db.db' # Path to database
) -> DBResources:
    "Initialize a database and return the database, as well as its videos, frames, runs, and runframes tables."
    db = database(path)
    videos = db.create(Video, transform=True)
    frames = db.create(Frame, transform=True, foreign_keys=[('video_id', 'video', 'id')])
    runs = db.create(Run, transform=True)
    runframes = db.create(RunFrame, pk=['run_id', 'frame_id', 'type'], foreign_keys=[('run_id', 'run', 'id'), ('frame_id', 'frame', 'id')], transform=True)
    return DBResources(db, videos, frames, runs, runframes)


In [ ]:
# !rm db.db
o = init_db()

In [ ]:
o.db

<Database <apsw.Connection "/app/data/vlm-monitor/nbs/db.db">>

In [ ]:
o.videos.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [video] (
   [id] INTEGER PRIMARY KEY,
   [title] TEXT,
   [overview] TEXT,
   [transcript] TEXT,
   [length] INTEGER,
   [sample_rate] INTEGER
)
```

</div>

## VLM

In [ ]:
#| export
from fastllm.types import Msg, Part, PartType

In [ ]:
?Msg

````python
def Msg(
    role:str, content:List
)->None:

````

````
A normalized message.
````

**File:** `/usr/local/lib/python3.12/site-packages/fastllm/types.py`; line: 57

**Type:** type

In [ ]:
?Part

````python
def Part(
    type:str, text:str=None, data:dict=None
)->None:

````

````
A normalized content part.
````

**File:** `/usr/local/lib/python3.12/site-packages/fastllm/types.py`; line: 23

**Type:** type

In [ ]:
?PartType

Init signature: PartType(*values)
Docstring:      An `ImportEnum` that behaves like a `str`
File:           /usr/local/lib/python3.12/site-packages/fastcore/basics.py
Type:           EnumType
Subclasses:     

In [ ]:
#| export
def user(
    txt:str,
    img:str|None=None,
)->Msg:
    "Build a user message with optional image."
    if img is None: return Msg(role='user', content=[Part(type=PartType.text, text=txt)])
    else: return Msg(role='user', content=[Part(type=PartType.input_image, text=img), Part(type=PartType.text, text=txt)])

In [ ]:
user('你好')

<div class="prose" markdown="1">

**Msg**

- role: `user`

<contents>

**Part** (`text`)

你好

<details markdown='1'>

- data: `None`

</details>

</contents>

</div>

In [ ]:
#| export
def assistant(
    txt:str,
    data:dict={'citations':[]},
)->Msg:
    "Build an assistant message."
    return Msg(role='assistant', content=[Part(type=PartType.text, text=txt, data=data)])

In [ ]:
assistant('嗨')

<div class="prose" markdown="1">

**Msg**

- role: `assistant`

<contents>

**Part** (`text`)

嗨

<details markdown='1'>

- data: `{'citations': []}`

</details>

</contents>

</div>

In [ ]:
from fastllm.acomplete import acomplete
?acomplete

````python
async def acomplete(
    msgs, model, api_name:NoneType=None, vendor_name:NoneType=None, api_key:NoneType=None, base_url:NoneType=None,
    xtra_body:NoneType=None, xtra_hdrs:NoneType=None, stream:bool=False, stop_callables:NoneType=None, retries:int=2,
    retry_delay:float=0.5, system:NoneType=None, max_tokens:NoneType=None, temperature:NoneType=None,
    tools:NoneType=None, tool_choice:NoneType=None, reasoning_effort:NoneType=None, web_search_options:NoneType=None
):

````

````
Unified completion across different APIs.
````

**File:** `/usr/local/lib/python3.12/site-packages/fastllm/acomplete.py`; line: 142

**Type:** function

In [ ]:
from cachy import enable_cachy, disable_cachy; enable_cachy()
await acomplete([user('hi')], 'deepseek-v4-flash', vendor_name='deepseek')

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

好的，用户只发了一个“hi”，这是非常简单的打招呼。用户可能刚进入对话，想测试我是否在线或者开始一个友好的交流。深层需求应该是希望得到热情、友好的回应，开启一次对话。我不需要复杂分析，直接礼貌问候并表达乐于助人的态度，用开放式的邀请让用户提出具体问题。想到了用“你好！”开头，加上表情符号显得亲切，然后自我介绍并说明能力范围，最后用提问引导对话继续。

</details>

你好！很高兴见到你！😊

有什么我可以帮你的吗？无论是回答问题、帮你整理信息、提供创作灵感，还是聊聊天，我都很乐意陪你一起。你只需告诉我需求，剩下的交给我！

<details markdown='1'>

- model: `deepseek-v4-flash`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=5, completion_tokens=143, total_tokens=148, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=97, raw={'prompt_tokens': 5, 'completion_tokens': 143, 'total_tokens': 148, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 97}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5})`

</details>

</div>

In [ ]:
from fastllm.types import Completion
async def stream(
    msgs:list|None=None,  # Messages to send
    model:str='',  # Model name (e.g. 'deepseek-v4-flash')
    max_think:float=float('inf'),  # Max thinking tokens to display
    usage:bool=True,  # Show usage info in output
    display:bool=True,  # Print text/thinking as it arrives
    **kwargs,  # Passed to acomplete
) -> Completion:  # Return the final completion
    "Stream a response, printing text/thinking as it arrives. Returns the final completion."
    assert msgs is not None, 'no messages provided'
    assert model!='', 'no model name provided'
    think_cnt, seen_txt = 0, False
    async for o in await acomplete(msgs, model, stream=True, **kwargs):
        if not isinstance(o, Completion) and display:
            if o.get('thinking') and think_cnt<max_think: print('🤔', end='', flush=True)
            if txt:=o.get('text'): print(f"{'\n\n' if not seen_txt else ''}{txt}", end='', flush=True) or not seen_txt and (seen_txt:=True)
            think_cnt+=1
    if display: print()
    return o

In [ ]:
r = await stream([user('hi')], 'deepseek-v4-flash', vendor_name='deepseek')

🤔

🤔

🤔

🤔

🤔

...

聊聊

，

都可以

告诉我

！

In [ ]:
#| export
from base64 import b64encode
def img2b64(
    path:Path,
)->str:
    "Encode an image file as a base64 data URL."
    return 'data:image/png;base64,'+b64encode(Path.read_bytes(path)).decode()

In [ ]:
r = await stream([user('what do ye elf eyes see', img2b64(Path('./test.jpg')))], 'bytedance-seed/seed-2.0-lite', vendor_name='openrouter', reasoning_effort='high')



*

til

ts

 point

y

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()